# Phase 1 — Nepali (Model L) corpus and tokenizer

Runs the Nepali half of the project on Kaggle **or** Colab, so it can proceed
in parallel with the Hindi (Model H) work happening elsewhere. The two models
are independent by construction — separate corpora, tokenizers, vocabularies —
so there is nothing to synchronise between them.

## Platform

The next cell detects Kaggle or Colab and configures paths accordingly.

| | Kaggle | Colab |
|---|---|---|
| RAM | 30 GB | ~13 GB |
| Session | 12 h | disconnects readily |
| Persistence | `/kaggle/working` (20 GB, same notebook) | Google Drive |

**Kaggle is preferable here.** The deduplication index holds one MinHash
signature per kept document and is the largest object in memory at corpus
scale, so RAM is the binding constraint.

### Kaggle settings (right-hand sidebar)

1. **Accelerator → None** — this stage is CPU and I/O bound; a GPU spends
   quota that Phase 2 needs.
2. **Internet → On** — off by default, and the most common cause of an
   unexplained dataset failure.
3. **Add-ons → Secrets** — add `GH_TOKEN` (classic token, `repo` scope).


## 1 · Detect the platform and set up paths


In [ ]:
import os, sys, subprocess, pathlib, shutil

ON_KAGGLE = pathlib.Path('/kaggle').exists()
ON_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').exists()

if ON_KAGGLE:
    PLATFORM = 'kaggle'
    WORK = pathlib.Path('/kaggle/working')
    # /tmp is wiped between sessions, which is what we want for the dataset
    # cache: it would otherwise consume the 20 GB persistent quota that has
    # to hold the corpus and checkpoints.
    CACHE = pathlib.Path('/tmp/hf_cache')
elif ON_COLAB:
    PLATFORM = 'colab'
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = pathlib.Path('/content/drive/MyDrive/lma_project')
    WORK.mkdir(parents=True, exist_ok=True)
    CACHE = pathlib.Path('/content/hf_cache')
else:
    PLATFORM = 'local'
    WORK = pathlib.Path.cwd()
    CACHE = pathlib.Path.home() / '.cache/huggingface'

REPO = WORK / 'lma'
os.environ['HF_HOME'] = str(CACHE)

print('platform :', PLATFORM)
print('work dir :', WORK)
print('cpus     :', os.cpu_count())
with open('/proc/meminfo') as fh:
    print(f'memory   : {int(fh.readline().split()[1]) / 1024**2:.1f} GB')
usage = shutil.disk_usage(WORK)
print(f'disk free: {usage.free / 1e9:.1f} GB')

probe = subprocess.run(['curl', '-sS', '-m', '10', '-o', '/dev/null',
                        '-w', '%{http_code}', 'https://huggingface.co'],
                       capture_output=True, text=True)
online = probe.stdout.strip() == '200'
print('internet :', 'ON' if online else 'OFF  <-- enable it in the settings panel')
assert online, 'Enable Internet, then re-run this cell.'


## 2 · Clone the repository

GitHub Classroom repos are private, so this needs a personal access token.
Create one at **Settings → Developer settings → Personal access tokens →
Tokens (classic)** with the `repo` scope, then store it as a secret:

- **Kaggle:** Add-ons → Secrets → `GH_TOKEN`
- **Colab:** 🔑 sidebar → Add new secret → `GH_TOKEN` (enable notebook access)

A *fine-grained* token defaults to no repository access and fails with a
misleading `403 Write access to repository not granted`.


In [ ]:
REPO_URL = 'https://github.com/CL3-410/individual-project-Karan-thayat.git'
BRANCH   = 'phase-1'

def read_secret(label):
    """Read a platform secret, returning None when it is unavailable."""
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(label)
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(label)
    except Exception:
        return None
    return os.environ.get(label)

GH_TOKEN = read_secret('GH_TOKEN')
HF_TOKEN = read_secret('HF_TOKEN')
print('GH_TOKEN:', 'set' if GH_TOKEN else 'MISSING - the clone will fail')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

if (REPO / '.git').exists():
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO), *args], check=False)
    print('updated existing clone')
else:
    # The token is injected only at call time and scrubbed from any output, so
    # it never appears in the notebook or its saved results.
    auth = REPO_URL.replace('https://', f'https://{GH_TOKEN}@') if GH_TOKEN else REPO_URL
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, auth, str(REPO)],
                       capture_output=True, text=True)
    out = r.stdout + r.stderr
    print((out.replace(GH_TOKEN, '***') if GH_TOKEN else out).strip() or 'cloned')

ok = (REPO / 'lma' / 'data' / 'sources.py').exists()
print('code present:', ok)
assert ok, 'Clone failed. Check GH_TOKEN has the classic `repo` scope.'


## 3 · Dependencies

No PyTorch needed for Phase 1. `sentencepiece` trains the tokenizer from
scratch and `datasets` is only a download client — neither is a pretrained
model, which the assignment forbids.


In [ ]:
%pip install -q sentencepiece datasets pyyaml beautifulsoup4 lxml regex 2>&1 | tail -2

import sentencepiece, datasets
print('sentencepiece', sentencepiece.__version__)
print('datasets     ', datasets.__version__)


## 4 · Scale the Nepali sources up

Two edits: widen the Sangraha shard pattern from one shard to all 25, and
drop the per-source document caps.

**Note the resource gap this exposes.** Sangraha carries 25 verified Nepali
shards against 100 for Hindi — a 4× difference in the largest open corpus for
each language, before any filtering. That gap *is* the phenomenon this project
studies, so it should be reported, not worked around.

The annotated original is kept as `*.sampled.yaml`, because rewriting YAML
discards comments and those comments record why each source is configured as
it is.


In [ ]:
import yaml

def scale_up(manifest_path, edits):
    """Widen shard globs and remove document caps for a full-scale run."""
    manifest_path = pathlib.Path(manifest_path)
    backup = manifest_path.with_suffix('.sampled.yaml')
    if not backup.exists():
        backup.write_text(manifest_path.read_text())
    cfg = yaml.safe_load(manifest_path.read_text())
    for src in cfg['sources']:
        if src['source_id'] not in edits:
            continue
        src.pop('limit', None)
        if edits[src['source_id']]:
            src['data_files'] = edits[src['source_id']]
    manifest_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
    return cfg

cfg = scale_up(REPO / 'nepali/configs/sources.yaml', {
    'sangraha_ne':  'verified/nep/*.parquet',
    'wikipedia_ne': None,
})
for s in cfg['sources']:
    if s.get('enabled', True):
        print(f"  {s['source_id']:30s} {s['collection']:10s} "
              f"{s.get('data_files') or s.get('config') or s.get('path', '')}")


## 5 · Build the Nepali corpus

**The long cell.** Nepali is the lower-resource language, so this exhausts its
sources rather than hitting the 500M target — expect 1–3 hours and a corpus
well short of Hindi's. That is the expected result, and the assignment permits
it explicitly: *"If the lower-resource language cannot reach 500M from public
sources, collect the maximum feasible amount, report the exact token count, and
justify the shortfall."*

### If the session ends first

Re-run **this same cell**. `--resume` reconciles the output files against the
checkpoint, restores the deduplication index, and skips what each source
already yielded. A resumed run has been verified to produce a byte-identical
corpus to an uninterrupted one, including after `kill -9`.

For an unattended run use **Save Version → Save & Run All (Commit)** on Kaggle;
an interactive session stops when the browser closes.


In [ ]:
def run(script, *extra, cwd=None):
    """Run a pipeline script, streaming its output live rather than buffering."""
    cmd = [sys.executable, str(REPO / 'scripts' / script), *extra]
    print(' '.join(cmd), '\n')
    proc = subprocess.Popen(cmd, cwd=str(cwd or REPO), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    return proc.wait()

rc = run('build_corpus.py',
         '--config',  str(REPO / 'nepali/configs/pipeline.yaml'),
         '--sources', str(REPO / 'nepali/configs/sources.yaml'),
         '--profile', 'full',
         '--target-tokens', '500000000',
         '--checkpoint-dir', str(WORK / 'checkpoints_ne'),
         '--checkpoint-every', '25000',
         '--resume',          # safe on a first run: no checkpoint means start fresh
         '--report-every', '20000')
print('\nexit code:', rc)


### Progress check

Run in a separate cell while the build is going, or in a new session to see
how far the previous one got.


In [ ]:
import json

state_file = WORK / 'checkpoints_ne' / 'ne_ingest_state.json'
if state_file.exists():
    st = json.load(open(state_file))
    print(f"segment     : {st['segments']}")
    print(f"docs seen   : {st['seen']:,}")
    print(f"docs kept   : {st['kept']:,}")
    print(f"train chars : {st['train_chars']:,}")
    print(f"elapsed     : {st['elapsed_seconds'] / 3600:.1f} h")
else:
    print('No checkpoint - the run has not saved one yet, or it finished and cleared it.')


## 6 · Train the Nepali tokenizer

Trained from scratch on the Nepali training split only. The vocabulary size is
**chosen empirically** rather than asserted: candidates are trained across
sizes and both algorithms, each measured on the same held-out validation text,
and the winner is the lowest fertility subject to an unknown-rate ceiling, with
near-ties resolved toward the smaller vocabulary to conserve embedding
parameters against Phase 2's fixed ~25M budget.


In [ ]:
rc = run('train_tokenizer.py',
         '--config', str(REPO / 'nepali/configs/pipeline.yaml'),
         '--profile', 'full', '--sweep', '--use-sweep-best')
print('\nexit code:', rc)

rc = run('tokenize_corpus.py', '--config', str(REPO / 'nepali/configs/pipeline.yaml'))
print('\nexit code:', rc)


## 7 · Verify

The self-tests include the Hindi-vs-Nepali separation checks, which matter
here: both languages are Devanagari, so a script check cannot tell them apart
and Hindi — far more abundant online — is the likely contaminant.


In [ ]:
print('=== component self-tests ===')
run('test_pipeline.py')

print('\n=== Nepali corpus purity ===')
sys.path.insert(0, str(REPO))
from lma.data.language_id import IndicLanguageIdentifier
from lma.data.splits import iter_jsonl

ident = IndicLanguageIdentifier()
train = REPO / 'nepali/data/processed/ne_train.jsonl'
if train.exists():
    counts = {}
    for i, row in enumerate(iter_jsonl(train)):
        if i >= 2000:
            break
        lang = ident.identify(row['text']).language
        counts[lang] = counts.get(lang, 0) + 1
    total = sum(counts.values()) or 1
    print('  verdicts:', counts)
    print(f"  identified as Nepali: {100 * counts.get('ne', 0) / total:.1f}%")
    print(f"  Hindi contamination : {100 * counts.get('hi', 0) / total:.1f}%")
else:
    print('  corpus not built yet')


## 8 · Manual-collection share

⚠️ **The requirement most likely to fail at full scale.** The assignment needs
**≥20% of training tokens** from manual collection, and removing the download
caps multiplies downloaded text while scraped volume stays fixed.

Unlike the token target, this floor has no shortfall clause — so if the number
below is under 20%, either collect more manual text (OCR is by far the highest
leverage: one 300-page book ≈ 80,000 words against ~900 for a scraped page) or
restore caps on the downloaded sources.


In [ ]:
stats = REPO / 'nepali/reports/corpus_stats.json'
if stats.exists():
    d = json.load(open(stats))
    share = d['collection_split']['manual_word_fraction']
    print(f"manual share: {100 * share:.1f}%  [{'OK' if share >= 0.20 else 'BELOW REQUIREMENT'}]")
    for src, words in sorted(d['by_source']['words'].items(), key=lambda kv: -kv[1]):
        print(f'  {src:32s} {words:>12,} words')
else:
    print('corpus statistics not written yet')


## 9 · Save the results

**Kaggle:** `/kaggle/working` persists between sessions of *this* notebook.
**Save Version → Save & Run All (Commit)** makes the outputs downloadable, and
the corpora can be published as a Kaggle Dataset (Output tab → New Dataset).

**Colab:** everything under `WORK` is already on Drive.

Either way, only the small artifacts belong in git — tokenizers, statistics,
the report. Corpora go to Drive with links in the README, as the assignment
requires.


In [ ]:
OUT = WORK / 'nepali_artifacts'
OUT.mkdir(exist_ok=True)
for sub in ['tokenizer', 'reports', 'configs']:
    src = REPO / 'nepali' / sub
    if src.exists():
        shutil.copytree(src, OUT / sub, dirs_exist_ok=True)

total = 0
for path in sorted(OUT.rglob('*')):
    if path.is_file():
        kb = path.stat().st_size / 1024
        total += kb
        print(f'{kb:9.1f} KB  {path.relative_to(OUT)}')
print(f'{total / 1024:9.1f} MB  TOTAL (small enough for git)')

print('\nCorpus files (large - Drive or Kaggle Dataset, not git):')
proc_dir = REPO / 'nepali/data/processed'
if proc_dir.exists():
    for f in sorted(proc_dir.glob('*.jsonl')):
        print(f'  {f.stat().st_size / 1e6:8.1f} MB  {f.name}')


### Push the small artifacts back to GitHub

Set `COMMIT = True` when you want this to run. Corpora stay out of git.


In [ ]:
COMMIT = False

if COMMIT and GH_TOKEN:
    subprocess.run(['git', '-C', str(REPO), 'config', 'user.name', 'Karan Thayat'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'config', 'user.email',
                    'karanthayat.30@gmail.com'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'add',
                    'nepali/reports', 'nepali/tokenizer', 'nepali/configs'], check=False)
    subprocess.run(['git', '-C', str(REPO), 'commit', '-m',
                    'Nepali: full-scale corpus statistics and tokenizer'], check=False)
    push = REPO_URL.replace('https://', f'https://{GH_TOKEN}@')
    r = subprocess.run(['git', '-C', str(REPO), 'push', push, BRANCH],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(GH_TOKEN, '***').strip())
else:
    print('COMMIT is False - nothing pushed.')
